In [ ]:
import pandas as pd

df = pd.read_csv('../data/train.csv')
df.head(10)
df.info()
df.describe()

In [ ]:
num_cols = df.select_dtypes(include=["int64", "float64"]).columns
cat_cols = df.select_dtypes(include=['object']).columns
print("Numerical columns:", num_cols, num_cols.size)
print("Categorical columns:", cat_cols, cat_cols.size)

In [ ]:

df.isna().sum().sort_values(ascending=False).head(20)

missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)

print(missing)


In [ ]:
import matplotlib.pyplot as plt

df['SalePrice'].hist(bins=100)
plt.xlabel('Sale Price')
plt.ylabel('Count')
plt.title('Distribution of Sale Prices')
plt.show()

In [ ]:
target_col = "SalePrice"

X = df.drop(columns=[target_col])
y = df[target_col]

numeric_cols = X.select_dtypes(include=["int64", "float64"]).columns
X_num = X[numeric_cols] 

from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy='median')
X_num_imputed = imputer.fit_transform(X_num)

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_num_imputed, y, test_size=0.2, random_state=42
)



In [ ]:
from sklearn.linear_model import LinearRegression

lin_reg = LinearRegression()
lin_reg.fit(X_train, y_train)

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np

y_pred = lin_reg.predict(X_test)

mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)

print("MAE:", mae)
print("RMSE:", rmse)


In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV

rf = RandomForestRegressor(random_state=42)

param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [None, 10, 20],
}

grid_search = GridSearchCV(
    rf,
    param_grid,
    cv=3,
    scoring="neg_mean_absolute_error",
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print("Best params:", grid_search.best_params_)
print("Best score (neg MAE):", grid_search.best_score_)

best_model = grid_search.best_estimator_

y_pred_rf = best_model.predict(X_test)

mae_rf = mean_absolute_error(y_test, y_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))

print("RF MAE:", mae_rf)
print("RF RMSE:", rmse_rf)
